# Детекция ботов по событиям внутри суточного окна

Задача: для каждой `cookie_id` из `data/test.csv` предсказать `score` от 0 до 1 — вероятность того, что кука принадлежит сервису автоматизированного сбора данных. Метрика — Precision при Recall ≥ 70% (`metric.py`).

Структура ноутбука:
1. Настройки и загрузка данных
2. Очистка данных
3. Исследование данных (EDA) и находки
4. Окно наблюдения
5. Признаки
6. Валидация и журнал экспериментов
7. Финальная модель и `submission.csv`

## 1. Настройки и загрузка данных

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from metric import precision_at_recall  # реализация метрики
SEED = 0
VALID_START = '2026-04-17'  # окна начиная с этой даты идут в валидацию

EVENTS_PATH = 'data/events.csv.gz'

In [2]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=date_cols)
test = pd.read_csv('data/test.csv', parse_dates=date_cols)
events = pd.read_csv(EVENTS_PATH, parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


## 2. Очистка данных

- `platform` записана в разном регистре (`android` / `Android` / `ANDROID`) и с синонимами (`iphone` = `ios`, `desktop` = `web` — одинаковые User-Agent). Приводим к трём значениям. Исходное написание сохраняем в `platform_raw` только для проверки в EDA.
- Строки файла событий перемешаны, поэтому сортируем по куке и времени: без этого паузы между событиями считаются неверно.

In [3]:
events['platform_raw'] = events.platform
events['platform'] = events.platform.str.lower().replace({'iphone': 'ios', 'desktop': 'web'})
events = events.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
dup_cols = [c for c in events.columns if c != 'platform_raw']
events['is_dup'] = events.duplicated(subset=dup_cols)
events = events[~events.is_dup].drop(columns=['is_dup'])

events.platform.value_counts()

platform
web        182943
android    124984
ios         16110
Name: count, dtype: int64

## 3. Исследование данных (EDA)

### 3.1. Типы событий, платформы, пропуски

In [4]:
print(events.event_name.value_counts(), '\n')
print(events.groupby('eid').event_name.unique(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))
print('количество дубликатов')
print(events.duplicated().sum())

event_name
item_view               118989
search_results_view      98928
photo_swipe              35962
favorite_add             18739
seller_page_view         17180
contact_phone_show       11154
captcha_shown             7829
login                     6177
contact_chat_open         6047
contact_message_sent      3032
Name: count, dtype: int64 

eid
100     [search_results_view]
200               [item_view]
210             [photo_swipe]
220        [seller_page_view]
300      [contact_phone_show]
301       [contact_chat_open]
303    [contact_message_sent]
400            [favorite_add]
500                   [login]
900           [captcha_shown]
Name: event_name, dtype: object 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.349
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x      

Пропуски объясняются типами событий: `search_query` и `search_page` заполнены только у поиска (~30% событий), `pointer_x/y` отсутствуют у мобильных приложений.

### 3.2. Капча: проверка на утечку

In [5]:
windows = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']],
    test[['cookie_id', 'window_start_ts', 'window_end_ts']],
])
cap = events[events.event_name == 'captcha_shown'].merge(windows, on='cookie_id')
cap['position'] = np.select(
    [cap.event_ts < cap.window_start_ts, cap.event_ts >= cap.window_end_ts],
    ['до окна', 'после окна'],
    default='внутри окна',
)
cap.position.value_counts()

position
после окна    7829
Name: count, dtype: int64

**Вывод:** все события `captcha_shown` лежат после окна наблюдения. Это информация из будущего (утечка), поэтому признаки по капче не используются. Фильтр по окну их и так отсекает.

### 3.3. Время создания куки: проверка на утечку

In [6]:
for name, df in [('train', train), ('test', test)]:
    print(name, (df.cookie_created_at >= df.window_end_ts).sum())

train 0
test 0


**Вывод:** у всех кук `cookie_created_at` раньше конца окна, поэтому возраст куки можно использовать как признак.

## 4. Окно наблюдения

In [7]:
def events_in_window(events, meta):
    """Оставляет события каждой куки внутри её окна: window_start_ts <= event_ts < window_end_ts."""
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]


ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

195482 88348


### 4.1. Боты и люди: сравнение кандидатов в признаки

In [8]:
def show_cookie(cookie_id):
    """Показывает события одной куки внутри окна по порядку времени."""
    return ev_tr[ev_tr.cookie_id == cookie_id].sort_values('event_ts')


show_cookie(train[train.target == 1].cookie_id.iloc[0])

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,platform_raw,window_start_ts,window_end_ts
207144,ck_e00e1e6c2a2ab4ac,2026-04-06 21:09:49,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,sankt-peterburg,NaN,сборка мебели,3.0,NaN,NaN,android,2026-04-06,2026-04-07
207145,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:36,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,uslugi,barnaul,NaN,мастер на все руки,5.0,NaN,NaN,android,2026-04-06,2026-04-07
207146,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:38,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,4.0,NaN,NaN,ANDROID,2026-04-06,2026-04-07
207147,ck_e00e1e6c2a2ab4ac,2026-04-06 21:39:45,300,contact_phone_show,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1024813.0,avtomobili,sankt-peterburg,pro,NaN,NaN,NaN,NaN,Android,2026-04-06,2026-04-07
207148,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:03,210,photo_swipe,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1021391.0,kvartiry_prodazha,sankt-peterburg,private,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
207149,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:17,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,NaN,sankt-peterburg,NaN,джинсы levis,4.0,NaN,NaN,ANDROID,2026-04-06,2026-04-07
207150,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:33,400,favorite_add,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1016533.0,zapchasti,sankt-peterburg,pro,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
207151,ck_e00e1e6c2a2ab4ac,2026-04-06 21:40:40,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1032690.0,uslugi,sankt-peterburg,pro,NaN,NaN,NaN,NaN,android,2026-04-06,2026-04-07
207152,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:32,200,item_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,1055889.0,zapchasti,sankt-peterburg,private,NaN,NaN,NaN,NaN,Android,2026-04-06,2026-04-07
207153,ck_e00e1e6c2a2ab4ac,2026-04-06 23:04:40,100,search_results_view,android,Avito/118.0 (Android 11; SM-A515F) okhttp/4.11.0,NaN,zapchasti,sankt-peterburg,NaN,шины зимние,2.0,NaN,NaN,Android,2026-04-06,2026-04-07


In [9]:
ev = ev_tr.sort_values(['cookie_id', 'event_ts']).copy()
ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()

cand = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'size'),
    first_ts=('event_ts', 'min'),
    last_ts=('event_ts', 'max'),
    gap_median=('gap', 'median'),
    gap_std=('gap', 'std'),
    gap_max=('gap', 'max'),
    max_page=('search_page', 'max'),
    n_queries=('search_query', 'nunique'),
    n_categories=('item_category', 'nunique'),
    n_locations=('item_location', 'nunique'),
    n_ua=('user_agent', 'nunique'),
).reset_index()
cand['duration_min'] = (cand.last_ts - cand.first_ts).dt.total_seconds() / 60
cand = cand.merge(train[['cookie_id', 'target']], on='cookie_id')

cand.drop(columns=['cookie_id', 'first_ts', 'last_ts']).groupby('target').median().T

target,0,1
n_events,12.0000,20.000000
gap_median,60.5000,25.000000
gap_std,1751.0893,1378.387444
gap_max,5894.0000,6053.000000
max_page,4.0000,7.000000
n_queries,3.0000,4.000000
n_categories,2.0000,2.000000
n_locations,5.0000,8.000000
n_ua,1.0000,1.000000
duration_min,138.7750,151.533333


**Выводы (медианы, 0 — люди, 1 — боты):**
- `gap_median`: 59 с против 25 с — боты кликают быстрее;
- `max_page`: 4 против 7 — боты листают выдачу глубже;
- `n_locations`: 5 против 8 — боты смотрят больше городов;
- `n_events`: 12 против 20 — боты делают больше действий;
- `n_categories`, `n_ua`, `duration_min`, `gap_max` почти не отличаются: боты тоже работают часами и делают длинные перерывы.

In [10]:
ev_t = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
pd.crosstab(ev_t.target, ev_t.event_name, normalize='index').T.round(3)

target,0,1
event_name,,
contact_chat_open,0.020,0.014
contact_message_sent,0.010,0.008
contact_phone_show,0.037,0.026
favorite_add,0.066,0.027
item_view,0.365,0.437
login,0.022,0.008
photo_swipe,0.123,0.070
search_results_view,0.305,0.352
seller_page_view,0.053,0.059


**Выводы:** люди чаще добавляют в избранное (6,6% против 2,7%), логинятся (2,1% против 0,8%) и листают фото (12,3% против 7,0%). Боты чаще просматривают объявления и выдачу.

In [11]:
chk = ev_tr.groupby('cookie_id').platform_raw.nunique().rename('n_platform_spellings').reset_index()
chk = chk.merge(train[['cookie_id', 'target']], on='cookie_id')
chk.groupby('target').n_platform_spellings.agg(
    mean='mean',
    share_multi=lambda s: (s > 1).mean(),
)

,mean,share_multi
target,,
0,3.193093,0.958203
1,3.507230,0.991101


**Вывод:** разнобой в написании `platform` есть у 96% людей и 99% ботов, то есть это шум логирования, а не сигнал. Значения нормализованы, признак из написаний не строится.

## 5. Признаки

In [12]:
def build_features(ev, meta):
    """Собирает по событиям окна одну строку признаков на каждую куку из meta."""
    ev = ev.sort_values(['cookie_id', 'event_ts']).copy()
    ev['gap'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()  # пауза до предыдущего события, сек
    ev['gap_short'] = ev.gap.where(ev.gap < 600) # пауза, только если меньше 10 минут, иначе NaN
    ev['is_fast'] = ev.gap < 3        # True, если пауза меньше 3 секунд
    ev['is_break'] = ev.gap > 1800    # True, если пауза больше 30 минут

    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        # объём и разнообразие
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
        'n_locations': g.item_location.nunique(),
        'n_queries': g.search_query.nunique(),
        # ритм
        'gap_median': g.gap.median(),
        'gap_short_mean': g.gap_short.mean(),
        'gap_short_std': g.gap_short.std(),
        'share_fast': g.is_fast.mean(),
        'n_sessions': g.is_break.sum()+1,
        # глубина поиска
        'max_page': g.search_page.max(),
    })

    # доли типов событий у куки
    share = pd.crosstab(ev.cookie_id, ev.event_name, normalize='index')
    share.columns = ['share_' + c for c in share.columns]
    f = f.join(share)

    # признаки из метаданных куки
    f = meta[['cookie_id', 'window_start_ts', 'cookie_created_at']].merge(
        f.reset_index(), on='cookie_id', how='left'
    )
    f['cookie_age_h'] = (f.window_start_ts - f.cookie_created_at).dt.total_seconds() / 3600
    f = f.drop(columns=['window_start_ts', 'cookie_created_at'])
    return f.fillna(-1)


X_tr = build_features(ev_tr, train)
X_te = build_features(ev_te, test)
y_tr = train.target.values
feature_cols = [c for c in X_tr.columns if c != 'cookie_id']

print(len(feature_cols), 'признаков')
X_tr.head()

20 признаков


,cookie_id,n_events,item_nunique,n_locations,n_queries,gap_median,gap_short_mean,gap_short_std,share_fast,n_sessions,...,share_contact_chat_open,share_contact_message_sent,share_contact_phone_show,share_favorite_add,share_item_view,share_login,share_photo_swipe,share_search_results_view,share_seller_page_view,cookie_age_h
0,ck_54a059eb7d3ea68b,7,4,5,3,21.0,27.833333,29.949402,0.142857,1,...,0.000000,0.000000,0.000000,0.000000,0.428571,0.000000,0.142857,0.428571,0.000000,3254.488611
1,ck_7e4de46eeab82974,41,19,12,5,57.5,46.617647,31.594989,0.024390,7,...,0.000000,0.000000,0.024390,0.024390,0.536585,0.024390,0.097561,0.195122,0.097561,4669.826667
2,ck_9320229ef6304522,36,19,9,10,22.0,28.642857,27.068109,0.027778,7,...,0.027778,0.027778,0.055556,0.083333,0.194444,0.083333,0.083333,0.361111,0.083333,791.866111
3,ck_30ccd25bc1714ed9,21,10,4,3,54.5,49.500000,26.535769,0.000000,3,...,0.047619,0.000000,0.047619,0.000000,0.666667,0.000000,0.000000,0.142857,0.095238,13.122222
4,ck_a77c5f05948cdeef,28,16,6,6,52.0,65.173913,53.287942,0.000000,4,...,0.000000,0.000000,0.035714,0.071429,0.428571,0.000000,0.142857,0.214286,0.107143,2276.090278


## 6. Валидация

Тест лежит позже трейна по времени, поэтому валидация тоже по времени: обучение на окнах до `VALID_START`, проверка на окнах начиная с этой даты.

In [13]:
def make_model():
    """Возвращает модель с фиксированными гиперпараметрами и seed."""
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1)


def evaluate(X, y, cols):
    """Обучает модель на ранних окнах и возвращает P@R0.7 на поздних."""
    is_valid = train.window_start_ts.ge(VALID_START).values
    model = make_model()
    model.fit(X.loc[~is_valid, cols], y[~is_valid])
    p_va = model.predict_proba(X.loc[is_valid, cols])[:, 1]
    return precision_at_recall(y[is_valid], p_va)

scores = []
for seed in [0, 1, 2, 3, 4]:
    SEED = seed
    scores.append(evaluate(X_tr, y_tr, feature_cols))
SEED = 0  # возвращаем исходный seed

print('среднее:', round(np.mean(scores), 4), 'разброс:', round(np.std(scores), 4), np.round(scores, 4))

is_valid = train.window_start_ts.ge(VALID_START).values
print('P@R0.7 на валидации:', round(evaluate(X_tr, y_tr, feature_cols), 4))
print('доля ботов (уровень константы):', round(y_tr[is_valid].mean(), 4))

среднее: 0.4452 разброс: 0.0188 [0.4392 0.4786 0.4291 0.4275 0.4516]
P@R0.7 на валидации: 0.4392
доля ботов (уровень константы): 0.082


### Журнал экспериментов

Модель: RandomForest (300 деревьев, `min_samples_leaf=3`, `random_state=0`).

| № | Шаг | Признаки | P@R0.7 |
|---|---|---|---|
| 0 | Константа (всем одинаковый score) | — | 0.0820 |
| 1 | Baseline из quickstart | n_events, item_nunique | 0.1025 |
| 2 | + ритм и поиск | + gap_median, max_page, n_locations, n_queries | 0.2258 |
| 3 | + доли типов событий | + share_* по event_name | 0.3019 |
| 4 | + возраст куки | + cookie_age_h | 0.3500 |
| 5 | + ритм внутри сессии | + gap_shor, is_fast, is break | 0.433 |
| 6 | (не используем) - коэффициент вариации пауз | - gap_short_cv | 0.4059 ± 0.009 |
| 7 | удаление дубликатов событий | те же признаки | 0.4427 ± 0.009  |
| 8 | пропуски - -1 вместо 0 | те же признаки | 0.4452 ± 0.019 (шум) |

## 7. Финальная модель и submission.csv

In [14]:
final_model = make_model()
final_model.fit(X_tr[feature_cols], y_tr)

sub = pd.DataFrame({
    'cookie_id': X_te.cookie_id,
    'score': final_model.predict_proba(X_te[feature_cols])[:, 1],
})


# проверки формата: каждая кука из test ровно один раз, score в [0, 1]
assert len(sub) == len(test)
assert sub.cookie_id.is_unique
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.between(0, 1).all()

sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.015778
1,ck_a76ee3b3e3e522fd,0.084757
2,ck_94c9a4d382689e82,0.023829
3,ck_8eaf9509ad9462a0,0.004250
4,ck_9a88a5a989cb5bc6,0.002665
